# Chapter 9: Unsupervised Learning

**Foundations of Data Science**

## In this lesson

1. No label, no right answer: what "good" means without an outcome
2. k-means: the algorithm in four lines, and what it assumes
3. Choosing k: elbow, silhouette, and judgment
4. Reading the clusters, then checking them against something you held back
5. PCA: fewer axes, most of the spread
6. Stability: would a different sample give the same clusters?
7. Hierarchical clustering and when to prefer it
8. Pitfalls

## The one idea

**Without a label there is no accuracy, so clusters are judged by whether they are stable and whether they are useful.** A clustering is a hypothesis about structure. You test it the same way as any hypothesis: does it hold up on new data, and does it predict something you didn't use to build it?

The clinic question changes shape. Suppose the clinic has no diabetes labels yet (a new site, a new outcome, a new population). **Can the screening measurements alone sort patients into groups that are worth treating differently?** We'll build the groups without the label, then reveal the label and see if the groups earned their keep.

## Setup

<!-- nb-links -->
[Download this notebook](chapter_09_unsupervised.ipynb?download=1){download="chapter_09_unsupervised.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/chapter_09/chapter_09_unsupervised.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/chapter_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("chapter_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this chapter's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

## 1. Features, and the first decision: scale

Clustering measures distance between people. Age runs 20 to 80; BMI 15 to 85; systolic pressure 70 to 230. Without rescaling, blood pressure would dominate every distance and the clusters would just be blood pressure bins. **Standardize** (subtract the mean, divide by the SD) so each feature contributes equally. This is a decision, not a default: if the clinic thinks waist matters twice as much as age, weight it twice.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler

nhanes = pd.read_parquet("data/processed/nhanes.parquet")
adults = nhanes[nhanes["age"] >= 20].copy()
adults["sex"] = adults["sex"].map({1: "male", 2: "female"})

features = ["age", "bmi", "waist_cm", "sbp", "dbp"]
clus = adults.dropna(subset=features).copy()      # complete cases only for clustering; note how many we lose
print("adults with all five measurements:", len(clus), "of", len(adults))

scaler = StandardScaler()
Z = scaler.fit_transform(clus[features])          # Z has mean 0, SD 1 per column
pd.DataFrame(Z, columns=features).describe().loc[["mean", "std"]].round(2)

## 2. k-means

The algorithm, in words:

1. Pick k points as starting centers (randomly).
2. Assign every person to the nearest center.
3. Move each center to the average of the people assigned to it.
4. Repeat 2 and 3 until nothing changes.

It minimizes **inertia**: the total squared distance from each person to their center. It assumes clusters are roughly round blobs of similar size; it will happily cut a long thin cloud into slices and call them clusters. Run it several times from different starts (`n_init`) and keep the best, because step 1 is random.

In [ ]:
from sklearn.cluster import KMeans

km3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(Z)
clus["cluster"] = km3.labels_
print("cluster sizes:", np.bincount(km3.labels_))
print("inertia:", round(km3.inertia_, 1))

## 3. Choosing k

Inertia always falls as k grows (k = n gives inertia 0), so you can't just minimize it. Two tools and a rule:

- **Elbow**: plot inertia against k and look for the bend where adding a cluster stops buying much.
- **Silhouette**: for each person, how close they are to their own cluster compared to the next nearest one, from -1 to 1. Average it; higher is better separated. Peaks are suggestive, not decisive.
- **The rule**: pick the smallest k the clinic can act on. Three groups a nurse can describe beats seven groups that a silhouette prefers by 0.01.

In [ ]:
from sklearn.metrics import silhouette_score

ks = range(2, 9)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Z, km.labels_, sample_size=3000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inertias, marker="o"); axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia"); axes[0].set_title("Elbow")
axes[1].plot(list(ks), sils, marker="o"); axes[1].set_xlabel("k"); axes[1].set_ylabel("mean silhouette"); axes[1].set_title("Silhouette")
for ax in axes: ax.grid(alpha=0.3)
plt.show()
print(pd.DataFrame({"k": list(ks), "inertia": np.round(inertias), "silhouette": np.round(sils, 3)}).to_string(index=False))

The elbow is soft (it usually is on real data) and the silhouette is modest everywhere, which tells you these are not crisp, well-separated groups; they're regions of a continuous cloud. That's normal for physiological data. We'll go with k = 3 because it's readable; the sections below would work the same for 4.

## 4. Reading the clusters, then checking them

Two steps, in this order. First describe each cluster in the original units so a human can name it. Then, and only then, bring in the thing you held back (diabetes) and see whether the clusters separate it. The order matters: if you peek at diabetes first, you'll describe the clusters in terms of it and fool yourself about what the measurements alone can do.

In [ ]:
# Cluster profiles in original units
profile = clus.groupby("cluster")[features].mean().round(1)
profile["n"] = clus["cluster"].value_counts().sort_index()
profile["share_male"] = clus.groupby("cluster")["sex"].apply(lambda s: (s == "male").mean()).round(2)
profile

In [ ]:
# Profiles as a picture: standardized means per cluster, so every feature is on the same scale
prof_z = pd.DataFrame(Z, columns=features).assign(cluster=km3.labels_).groupby("cluster").mean()
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.heatmap(prof_z, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1.2, vmax=1.2, ax=ax, cbar_kws={"label": "SD from overall mean"})
ax.set_title("What makes each cluster different (standardized means)"); ax.set_ylabel("cluster")
plt.show()

Name them from the heatmap before reading on. One cluster is young and lean with low pressure; one is older with high pressure and average weight; one is heavy (high BMI and waist) with moderate pressure. Those are descriptions a nurse could use. Now the held-back check.

In [ ]:
# The reveal: diabetes prevalence by cluster. The clustering never saw this column.
check = clus.groupby("cluster")["diabetes"].agg(["mean", "count"]).round(3)
check.columns = ["diabetes prevalence", "n"]
check["vs overall"] = (check["diabetes prevalence"] / clus["diabetes"].mean()).round(2)
check

The young, lean cluster has 4% diabetes prevalence; the other two have 22% each, nearly six times higher, and the clustering never saw the label. That's the evidence it captured something real. Note also what the two high-risk clusters say: there are two routes to the same prevalence, one through age and pressure, one through weight. A single risk score would blur that; the clusters keep it visible, which is the kind of thing a clinic can act on differently (one group needs blood pressure management, the other weight management). It is weaker than a supervised model (chapter 7 used the label and got AUC 0.80), which is the expected trade: unsupervised structure is coarser, but it's available when there's no label, and it generalizes to outcomes you haven't measured yet.

## 5. PCA: fewer axes, most of the spread

Five features is five dimensions, which you can't draw. **Principal component analysis** finds new axes, each a weighted combination of the originals, ordered by how much of the spread in the data they capture. The first two usually carry most of it, and plotting those two is the standard way to *see* a clustering.

Plainly: imagine the cloud of 4,400 people in five dimensions. PC1 is the direction the cloud is longest in. PC2 is the longest direction at right angles to PC1. And so on. The **loadings** say what each PC is made of in terms of the original features.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(Z)
explained = pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(len(features))])
print("share of variance explained:\n", explained.round(3).to_string())
print("\ncumulative:", explained.cumsum().round(3).to_dict())

loadings = pd.DataFrame(pca.components_.T, index=features, columns=explained.index).round(2)
loadings[["PC1", "PC2"]]

In [ ]:
# The clusters in PCA space
pcs = pca.transform(Z)
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(pcs[:, 0], pcs[:, 1], c=km3.labels_, cmap="Set1", s=8, alpha=0.5)
centers = pca.transform(km3.cluster_centers_)
ax.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=150, label="cluster centers")
ax.set_xlabel(f"PC1 ({explained['PC1']:.0%} of variance): " + ", ".join(f"{f} {w:+.2f}" for f, w in loadings["PC1"].items() if abs(w) > 0.3))
ax.set_ylabel(f"PC2 ({explained['PC2']:.0%}): " + ", ".join(f"{f} {w:+.2f}" for f, w in loadings["PC2"].items() if abs(w) > 0.3))
ax.set_title("Three clusters in the first two principal components"); ax.legend()
plt.show()

PC1 is a "size" axis (BMI and waist load together) and PC2 is an "age and pressure" axis. The two together carry most of the spread, and the three clusters are three regions of one continuous cloud, boundaries drawn by k-means rather than found in the data. That's what the modest silhouette was saying.

PCA has a second use beyond pictures: as a preprocessing step to shrink many correlated features (hundreds of lab values, say) into a handful of components before clustering or modeling. The cost is interpretability; "PC3" is not a thing a clinician can measure.

## 6. Stability

If the clusters are real, a different sample of adults should produce nearly the same groups. Fit on a random half, fit on the other half, and compare how often pairs of people end up together under both fits. The **adjusted Rand index** does that comparison: 1 is identical partitions, 0 is what chance would give.

In [ ]:
from sklearn.metrics import adjusted_rand_score

rng = np.random.default_rng(0)
scores = []
for rep in range(10):
    idx = rng.permutation(len(Z))
    a, b = idx[: len(Z) // 2], idx[len(Z) // 2 :]
    km_a = KMeans(n_clusters=3, n_init=10, random_state=rep).fit(Z[a])
    km_b = KMeans(n_clusters=3, n_init=10, random_state=rep).fit(Z[b])
    # label everyone with both models and compare the two labelings
    scores.append(adjusted_rand_score(km_a.predict(Z), km_b.predict(Z)))
print("ARI across 10 split-half comparisons:", np.round(scores, 2))
print("mean:", np.mean(scores).round(2))

An ARI well above 0.5 across repeats means the three groups are reproducible, not an artifact of one random start. Report this number with any clustering you put in front of someone; a clustering with ARI 0.2 is a story about one dataset.

## 7. Hierarchical clustering

k-means needs k up front. **Agglomerative** clustering starts with every person as their own cluster and merges the two closest at each step until one cluster remains; the record of merges is a **dendrogram**, and you cut it at any height to get any k. Useful when you want to *see* the structure before choosing k, and when clusters aren't round. Slower (it compares every pair), so on a sample here.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

sample = rng.choice(len(Z), size=600, replace=False)
link = linkage(Z[sample], method="ward")   # Ward: merge the pair that increases inertia least; the hierarchical cousin of k-means

fig, ax = plt.subplots(figsize=(12, 4))
dendrogram(link, no_labels=True, color_threshold=0, ax=ax)
ax.set_title("Ward dendrogram on 600 adults: cut height = number of clusters"); ax.set_ylabel("merge distance")
plt.show()

# Cut at 3 and compare with k-means on the same people
h3 = fcluster(link, t=3, criterion="maxclust")
print("agreement with k-means (ARI):", round(adjusted_rand_score(h3, km3.labels_[sample]), 2))

Moderate agreement, not high. Two reasonable algorithms drew different boundaries through the same continuous cloud, which is one more reminder that the boundaries are ours, not the data's. When two methods disagree this much, report the clusters as regions of a continuum, not as types.

## 8. Pitfalls

- **Forgetting to scale.** The single most common error. Check `describe()` on the scaled matrix.
- **Categorical features.** Euclidean distance on one-hot columns is dubious. Either cluster on numeric features only (as here), use a distance built for mixed data (Gower), or cluster within categories.
- **Trusting the silhouette.** It rewards round, well-separated blobs. Real data rarely has those; a low silhouette doesn't mean the clusters are useless, and a high one doesn't mean they matter.
- **Describing clusters with the thing you want to predict.** Describe first, reveal second, as in section 4.
- **Outliers.** k-means centers get dragged by extreme values; the BMI 86 from chapter 2 pulls on its cluster. Check who's at the edges.
- **Reporting a clustering without a stability number.**

## Looking ahead
- **Reading**: Bruce & Bruce chapter 7 (unsupervised learning). For next chapter, Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, chapters 2 and 3 (free online). Time series: the rows are no longer independent.